In [1]:
import torch
from modelscope import snapshot_download, AutoModel, AutoTokenizer
import os

# 清除所有可能的代理环境变量
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)
os.environ.pop('HTTP_PROXY', None)
os.environ.pop('HTTPS_PROXY', None)

model_dir = snapshot_download('LLM-Research/Llama3-8B-Chinese-Chat', cache_dir='/home/nanyi/chat_wukong/model', revision='master')

/home/nanyi/miniconda3/envs/chat_wukong/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import gc
import pandas as pd  # <--- 补全导入
from datasets import Dataset # <--- 补全导入
from transformers import (
    BitsAndBytesConfig, 
    AutoModelForCausalLM, 
    AutoTokenizer, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, TaskType, get_peft_model

# --- 1. 环境清理与精度检测 ---
gc.collect()
torch.cuda.empty_cache()
compute_dtype = torch.bfloat16

# --- 2. 量化配置 ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model_path = '/home/nanyi/chat_wukong/model/LLM-Research/Llama3-8B-Chinese-Chat'

# --- 3. 加载模型 ---
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
)

# 彻底跳过 prepare_model_for_kbit_training
# A. 确保非 4-bit 层保持在 compute_dtype
for name, param in model.named_parameters():
    if param.dtype == torch.float16 or param.dtype == torch.bfloat16:
        param.data = param.data.to(compute_dtype)

# B. 开启输入梯度要求
model.enable_input_require_grads()

# C. 启用梯度检查点
model.gradient_checkpointing_enable()

# D. 禁用 KV Cache
model.config.use_cache = False

print("模型配置完成！")
print(f"当前空闲显存: {torch.cuda.mem_get_info(0)[0]/1024**2:.2f} MB")

# --- 4. Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# --- 5. 数据处理 ---
# 定义预处理函数
def process_func(example):
    MAX_LENGTH = 128  
    # 自动识别列名并拼接文本
    text = ""
    if 'instruction' in example and 'output' in example:
        input_val = example.get('input', '')
        text = f"### Instruction:\n{example['instruction']}\n### Input:\n{input_val}\n### Response:\n{example['output']}"
    elif 'content' in example:
        text = example['content']
    elif 'text' in example:
        text = example['text']
    else:
        # 兜底：拼接所有字符串类型的值
        text = " ".join([str(v) for v in example.values() if isinstance(v, str)])

    # Tokenize
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_tensors=None
    )
    
    # 设置 labels (用于计算损失)
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

print(" 正在加载数据集...")
# 读取数据 
df = pd.read_json('悟空_孙悟空对话_规范.json')
print(f"读取到 {len(df)} 条数据")

ds = Dataset.from_pandas(df)
tokenized_id = ds.map(process_func, remove_columns=ds.column_names, num_proc=4)
print(f" 数据处理完成！")

# --- 6. LoRA 配置 ---
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

# --- 7. 训练参数 ---
args = TrainingArguments(
    output_dir="./output/llama3_1_instruct_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    num_train_epochs=3,
    save_steps=100,
    learning_rate=1e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none",
    dataloader_pin_memory=False,
)

# --- 8. 启动训练 ---
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_id,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

print("开始训练...")
trainer.train()

Loading checkpoint shards: 100%|██████████| 4/4 [00:11<00:00,  2.89s/it]


模型配置完成！
当前空闲显存: 1567.00 MB
 正在加载数据集...
读取到 73 条数据


Map (num_proc=4): 100%|██████████| 73/73 [00:00<00:00, 619.31 examples/s]


 数据处理完成！
trainable params: 20,971,520 || all params: 8,051,232,768 || trainable%: 0.2605
开始训练...


/home/nanyi/miniconda3/envs/chat_wukong/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,4.492700
10,3.488400
15,2.997000
20,3.048800
25,2.965300
30,2.913900
35,2.848000
40,2.466200
45,2.083000
50,2.062300


/home/nanyi/miniconda3/envs/chat_wukong/lib/python3.12/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in /home/nanyi/chat_wukong/model/LLM-Research/Llama3-8B-Chinese-Chat - will assume that the vocabulary was not modified.
  warnings.warn(
/home/nanyi/miniconda3/envs/chat_wukong/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/nanyi/miniconda3/envs/chat_wukong/lib/python3.12/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in /home/nanyi/chat_wukong/model/LLM-Research/Llama3-8

TrainOutput(global_step=108, training_loss=2.1268698197823985, metrics={'train_runtime': 93.0933, 'train_samples_per_second': 2.352, 'train_steps_per_second': 1.16, 'total_flos': 502263260061696.0, 'train_loss': 2.1268698197823985, 'epoch': 2.958904109589041})

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from peft import PeftModel

mode_path = '/home/nanyi/chat_wukong/model/LLM-Research/Llama3-8B-Chinese-Chat'
lora_path = '/home/nanyi/chat_wukong/output/llama3_1_instruct_lora/checkpoint-100'

# 1. 定义与训练时完全一致的量化配置
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, 
    bnb_4bit_use_double_quant=True,
)

# 2. 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(mode_path, trust_remote_code=True)

# 3. 加载模型 
print("正在加载 4-bit 量化模型...")
model = AutoModelForCausalLM.from_pretrained(
    mode_path,
    quantization_config=bnb_config, 
    device_map="auto",
    trust_remote_code=True,
    # torch_dtype=torch.bfloat16, # 当使用 quantization_config 时，torch_dtype 通常由 bnb_config 控制，这里可以省略或保持一致
).eval()

# 4. 加载 LoRA 权重
print(f"正在加载 LoRA 权重 from {lora_path} ...")
model = PeftModel.from_pretrained(model, model_id=lora_path)
# 如果需要合并权重（可选，用于加速推理，但会占用更多显存）
# model = model.merge_and_unload() 

prompt = "孙悟空,我要开始念咒了"

messages = [
    {"role": "system", "content": "假设你是孙悟空。"},
    {"role": "user", "content": prompt}
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

model_inputs = tokenizer([input_text], return_tensors="pt").to(model.device)

print("生成中...")
generated_ids = model.generate(
    model_inputs.input_ids, 
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print('唐僧：', prompt)
print('孙悟空：', response)

/home/nanyi/miniconda3/envs/chat_wukong/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


正在加载 4-bit 量化模型...


Loading checkpoint shards: 100%|██████████| 4/4 [00:08<00:00,  2.10s/it]


正在加载 LoRA 权重 from /home/nanyi/chat_wukong/output/llama3_1_instruct_lora/checkpoint-100 ...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token.As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


生成中...
唐僧： 孙悟空,我要开始念咒了
孙悟空： 你念咒吧！我不怕！如果你能念得动我的金箍棒，那就再好不过了！快点念吧！别等了！别等了！
